In [2]:
# lectura de los datos

import pandas as pd
import read_data

# Suponiendo que el activo tiene ese nombre
# solo es remplazar por el nombre que se tenga
# solo se acepta archivos .parquet y .csv
nombre_activo: str = "EURUSD.parquet"
data: pd.DataFrame = read_data.read_asset(nombre_activo)

primer_lunes = data[data.index.dayofweek == 0].index[0]

data = data[primer_lunes:]

In [5]:
import find_best
import keys
import pandas as pd
import read_data
from use_tecnics import simple_methods, main

keys.candles = 20
keys.methods = simple_methods

ventana_entrenamiento = pd.Timedelta(weeks=3) + pd.Timedelta(days=4)
inicio_testeo = pd.Timedelta(days=3)
fin_testeo = pd.Timedelta(days=4)

inicio_train = data[data.index.dayofweek == 0].index[0].normalize()
fin_datos = data.index[-1]

signals_and_prices = None

while True:
    train_time = inicio_train + ventana_entrenamiento
    inicio_test = train_time + inicio_testeo
    fin_test = inicio_test + fin_testeo

    if fin_test >= fin_datos:
        break

    print(f"Entrenamiento: {inicio_train.strftime('%Y-%m-%d')} a {train_time.strftime('%Y-%m-%d')}")

    sub_data = data[inicio_train: train_time]
    best_ma = find_best.opti_main(sub_data)

    minutos_de_colchon = best_ma[1] * best_ma[2] * (1 if best_ma[0] in simple_methods else 8)
    tiempo_colchon = train_time - pd.Timedelta(minutes=minutos_de_colchon)
    
    bloque_completo = pd.concat([data[tiempo_colchon: train_time - pd.Timedelta(minutes=1)],
                                 data[inicio_test: fin_test]])

    velas_cerradas = read_data.ohlc_form(bloque_completo, f"{best_ma[1]}min")["close"]

    señales_generadas = main(best_ma[0], velas_cerradas, best_ma[2:])

    test_signals_only = señales_generadas.loc[inicio_test : fin_test]

    if test_signals_only["Signals"][test_signals_only.index[0]] == 1:
        start = test_signals_only.index[0]
    else:
        start = test_signals_only.index[1]

    if test_signals_only["Signals"][test_signals_only.index[-1]] == -1:
        start = test_signals_only.index[-1]
    else:
        start = test_signals_only.index[-2]
    
    if signals_and_prices is None:
        signals_and_prices = test_signals_only
    else:
        signals_and_prices = pd.concat([signals_and_prices, test_signals_only])

    inicio_train = inicio_train + pd.Timedelta(days=7)

Entrenamiento: 2025-02-24 a 2025-03-21
Resultado obtenido entrenando desde 2025-02-24 hasta 2025-03-20
Método: MIDPOINT, Datos optimizados [np.int64(16), np.int64(66)]

hit ratio: 0.7580645161290323
risk reward: 0.5694704136459483
profit factor: 2.4331917673963246
trades: 62
Resultado de sobre ajuste 1.2667466203133027
Entrenamiento: 2025-03-03 a 2025-03-28
Resultado obtenido entrenando desde 2025-03-03 hasta 2025-03-27
Método: MIDPOINT, Datos optimizados [np.int64(16), np.int64(66)]

hit ratio: 0.746031746031746
risk reward: 0.6113518030468577
profit factor: 2.210271903323255
trades: 63
Resultado de sobre ajuste 2.070716958590435
Entrenamiento: 2025-03-10 a 2025-04-04
Resultado obtenido entrenando desde 2025-03-10 hasta 2025-04-03
Método: MIDPOINT, Datos optimizados [np.int64(16), np.int64(66)]

hit ratio: 0.7222222222222222
risk reward: 0.5711431990103286
profit factor: 1.9799630899024727
trades: 72
Resultado de sobre ajuste 0.8663408862261022
Entrenamiento: 2025-03-17 a 2025-04-11
R

KeyboardInterrupt: 

In [7]:
signals_and_prices.to_parquet("Data/señales_por_un_mes.parquet")

In [9]:
signals_and_prices = pd.read_parquet("Data/señales_por_un_mes.parquet")
signals_and_prices

,Signals,Prices
time,,
2025-03-24 00:00:00,1,1.081470
2025-03-24 02:40:00,-1,1.083565
2025-03-24 04:16:00,1,1.082750
2025-03-24 09:04:00,-1,1.083590
2025-03-24 12:00:00,1,1.083450
...,...,...
2025-04-17 12:32:00,-1,1.137660
2025-04-17 12:48:00,1,1.137350
2025-04-17 17:04:00,-1,1.137660


In [10]:
from tester import backtest

backtest(signals_and_prices, False)

(np.float64(0.847457627118644),
 np.float64(1.2486587648159952),
 np.float64(6.936993137866641),
 59)

In [20]:
initial_mon = 100
cantidad = 0

for index in signals_and_prices.index:
    precio_actual = signals_and_prices["Prices"][index]
    current_signal = signals_and_prices["Signals"][index]

    print("Inicia con", initial_mon)
    if current_signal == 1 and cantidad == 0:
        cantidad = int(initial_mon / precio_actual)

        initial_mon -= (precio_actual * cantidad) 

    elif current_signal == -1 and cantidad > 0:

        initial_mon += (precio_actual * cantidad) 
        cantidad = 0

    print("Termina con", initial_mon)

Inicia con 100
Termina con 0.5047600000000045
Inicia con 0.5047600000000045
Termina con 100.19274000000001
Inicia con 100.19274000000001
Termina con 0.579740000000001
Inicia con 0.579740000000001
Termina con 100.27002
Inicia con 100.27002
Termina con 0.5926199999999966
Inicia con 0.5926199999999966
Termina con 100.29025999999999
Inicia con 100.29025999999999
Termina con 0.6662199999999956
Inicia con 0.6662199999999956
Termina con 100.00229999999999
Inicia con 100.00229999999999
Termina con 0.6395399999999825
Inicia con 0.6395399999999825
Termina con 100.07589999999998
Inicia con 100.07589999999998
Termina con 0.6997999999999678
Inicia con 0.6997999999999678
Termina con 100.05335999999997
Inicia con 100.05335999999997
Termina con 0.7375199999999609
Inicia con 0.7375199999999609
Termina con 100.09797999999995
Inicia con 100.09797999999995
Termina con 0.8437799999999385
Inicia con 0.8437799999999385
Termina con 100.15409999999994
Inicia con 100.15409999999994
Termina con 0.921059999999954